# Création des chroniques horaires 2040-2060

Construit, pour chaque technologie renouvelable, un CSV horaire 2040-2060 avec une colonne par pays
(`FR, DE, CH, IT, NL, BE, ES, PT, UK, IE`).

Chaque famille de technologies a sa propre logique de construction (cf. sous-sections) :

- **Solaire, éolien, run-of-river, lake inflows, PHS inflows** : reconstruits directement à partir des
  données brutes PECD4.2 (scénario SSP245), année par année sur 2040-2060.
- **Contraintes lacs (minimal outflow/volume, spill) et autres renouvelables (marine, biomass_coge,
  geothermal_coge, ocgt_coge, waste)** : la donnée source n'existe que pour la France sur une année de
  référence (2006) ; elle est recopiée sur les autres pays (0 pour les contraintes de lacs, valeur FR
  répliquée pour les autres renouvelables) puis dupliquée sur 2040-2060.

Les fichiers sont écrits dans un nouveau dossier `created_time_varying_inputs/` (séparé de
`inputs/time_varying_inputs/` pour ne rien écraser automatiquement).

In [54]:
import pandas as pd
import numpy as np
import re
import os
import glob

In [89]:
COUNTRIES = ["FR", "DE", "CH", "IT", "NL", "BE", "ES", "PT", "UK", "IE"]
YEARS = range(2040, 2061)
clim = 245

PECD_ROOT = f"raw_data/pecd42/SSP{clim}"
OUTPUT_DIR = f"created_time_varying_inputs/SSP{clim}"

os.makedirs(os.path.join(OUTPUT_DIR, "renew_profiles"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "hydro_profiles"), exist_ok=True)

## Fonctions communes

In [56]:
def group_and_average_columns(df, initials):
    """Moyenne des colonnes de zones PECD par pays (préfixe de 2 lettres) -- pour les facteurs de charge."""
    grouped = {}
    for initial in initials:
        matching = [c for c in df.columns if re.match(f"^{initial}", c)]
        if matching:
            grouped[initial] = df[matching].mean(axis=1)
    return pd.DataFrame(grouped)


def group_and_sum_columns(df, initials):
    """Somme des colonnes de zones PECD par pays -- pour les volumes d'énergie (hydro)."""
    grouped = {}
    for initial in initials:
        matching = [c for c in df.columns if re.match(f"^{initial}", c)]
        if matching:
            grouped[initial] = df[matching].sum(axis=1)
    return pd.DataFrame(grouped)


def expand_weekly_to_hourly(df):
    """Étale uniformément chaque valeur hebdomadaire PECD (NRG) sur les 168 heures de la semaine."""
    hourly_index = pd.date_range(
        start=df.index[0],
        end=df.index[-1] + pd.Timedelta(weeks=1) - pd.Timedelta(hours=1),
        freq="h",
    )
    expanded = pd.DataFrame(index=hourly_index)
    for col in df.columns:
        expanded[col] = df[col].reindex(df.index.repeat(168)).values / 24 / 7
    return expanded


def extend_to_yearend(df, end):
    """Les séries hydro hebdomadaires PECD ne tombent pas toujours pile sur le 31/12 : on complète
    en répétant la dernière semaine disponible jusqu'à `end`."""
    if df.index.max() >= end:
        return df[df.index <= end]
    extension_index = pd.date_range(start=df.index.max() + pd.Timedelta(hours=1), end=end, freq="h")
    tail = df.iloc[-len(extension_index):].copy()
    tail.index = extension_index
    return pd.concat([df, tail])


def read_pecd_hydro_series(category, code, y, root=PECD_ROOT):
    """Les fichiers hydro PECD sont hebdomadaires et démarrent fin décembre de l'année y-1 (sauf
    quelques années qui démarrent pile au 1er janvier) : on essaie les deux formes de nom de fichier."""
    pattern = os.path.join(
        root, "hydro", category,
        f"P_CMI6_MPI-_MEHR_{code}_0000m_Pecd_SZON_S{y-1}*_E{y}*_NRG_TIM_07d_NA-_noc_org_NA_SP{clim}_NA---_StRnF_PECD4.2_fv1.csv",
    )
    matches = glob.glob(pattern)
    if not matches:
        pattern = os.path.join(
            root, "hydro", category,
            f"P_CMI6_MPI-_MEHR_{code}_0000m_Pecd_SZON_S{y}*_E{y}*_NRG_TIM_07d_NA-_noc_org_NA_SP{clim}_NA---_StRnF_PECD4.2_fv1.csv",
        )
        matches = glob.glob(pattern)
    if not matches:
        raise FileNotFoundError(f"Aucun fichier {category} trouvé pour {y}")
    return pd.read_csv(matches[0], index_col=0, skiprows=52, parse_dates=True)


def fr_only_to_countries(path, countries, years, time_scale="hourly"):
    """Charge un profil de référence France-only et le réplique à l'identique sur chaque pays.

    `years` sert uniquement quand le fichier source ne porte pas l'année dans son index
    (ex: 'MM-DD' au lieu de 'YYYY-MM-DD') : le profil de référence est alors dupliqué sur ces années.
    """
    if time_scale == "daily":
        df = pd.read_csv(path, index_col=0)
        if len(df.index[0]) > 5:
            df.index = pd.to_datetime(df.index)
        else:
            parts = []
            for y in years:
                tmp = df.copy()
                tmp.index = pd.to_datetime(tmp.index.map(lambda d: f"{y}-{d}"), format="%Y-%m-%d")
                parts.append(tmp)
            df = pd.concat(parts)
    else:
        df = pd.read_csv(path, index_col=[0, 1])
        if len(df.index[0][0]) > 5:
            df.index = pd.to_datetime([pd.Timestamp(d) + pd.Timedelta(hours=h) for d, h in df.index])
        else:
            parts = []
            for y in years:
                tmp = df.copy()
                tmp.index = pd.to_datetime([f"{y}-{d} {h}:00" for d, h in tmp.index], format="%Y-%m-%d %H:%M")
                parts.append(tmp)
            df = pd.concat(parts)

    values = df.iloc[:, 0].values
    return pd.DataFrame({c: values for c in countries}, index=df.index)


def fr_only_other_zero(df, keep="FR"):
    """Met à 0 tous les pays sauf `keep` (utilisé pour les contraintes de lacs, qui n'existent qu'en France)."""
    out = df.copy()
    for c in out.columns:
        if c != keep:
            out[c] = 0
    return out


def change_year(df, target_year):
    """Recopie un profil de référence sur une autre année (même jour/heure, année remplacée)."""
    out = df.copy()
    out.index = out.index.map(lambda ts: ts.replace(year=target_year))
    return out


def save(df, subfolder, name):
    """Réordonne les colonnes sur COUNTRIES (0 pour un pays absent, ex: CH pour l'éolien offshore) et écrit le CSV."""
    df = df.copy()
    for c in COUNTRIES:
        if c not in df.columns:
            df[c] = 0.0
    df = df[COUNTRIES]
    df.index.name = "Date"
    path = os.path.join(OUTPUT_DIR, subfolder, f"{name}.csv")
    df.to_csv(path)
    print(f"{name}: {df.shape[0]} lignes, {df.index.min()} -> {df.index.max()}")
    return df

## Solaire — pv_roof_com, pv_roof_indiv, pv_ground

Fichiers PECD4.2 SPV (NUT0) par année, un fichier par sous-technologie (suffixe `_60_` = toiture
commerciale, `_61_` = toiture résidentielle, `_62_` = centrale au sol).

In [57]:
def load_pecd_solar(suffix, y):
    path = os.path.join(
        PECD_ROOT, "solar",
        f"P_CMI6_MPI-_MEHR_SPV_0000m_Pecd_NUT0_S{y}01010000_E{y}12312300_CFR_TIM_01h_NA-_noc_org_{suffix}_SP{clim}_NA---_PhM03_PECD4.2_fv1.csv",
    )
    return pd.read_csv(path, index_col=0, skiprows=52, parse_dates=True)


pv_roof_com_all, pv_roof_indiv_all, pv_ground_all = [], [], []
for y in YEARS:
    pv_roof_com_all.append(group_and_average_columns(load_pecd_solar(60, y), COUNTRIES))
    pv_roof_indiv_all.append(group_and_average_columns(load_pecd_solar(61, y), COUNTRIES))
    pv_ground_all.append(group_and_average_columns(load_pecd_solar(62, y), COUNTRIES))

pv_roof_com = save(pd.concat(pv_roof_com_all), "renew_profiles", "pv_roof_com")
pv_roof_indiv = save(pd.concat(pv_roof_indiv_all), "renew_profiles", "pv_roof_indiv")
pv_ground = save(pd.concat(pv_ground_all), "renew_profiles", "pv_ground")

pv_roof_com: 184104 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00
pv_roof_indiv: 184104 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00
pv_ground: 184104 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00


## Éolien terrestre — onshore

Deux fichiers PECD par année : parc existant (`_30_`) et nouveau parc (`_34_`). On combine
`0.9 * new + 0.1 * existing`, puis on recale la moyenne FR sur la référence CLEVER (0.28).

In [90]:
def load_pecd_onshore(suffix, y):
    path = os.path.join(
        PECD_ROOT, "wind", "onshore",
        f"P_CMI6_MPI-_MEHR_WON_NA---_Pecd_PEON_S{y}01010000_E{y}12312300_CFR_TIM_01h_NA-_noc_org_{suffix}_SP{clim}_ReGrB_PhM04_PECD4.2_fv1.csv",
    )
    return pd.read_csv(path, index_col=0, skiprows=52, parse_dates=True)


onshore_existing_all, onshore_new_all = [], []
for y in YEARS:
    onshore_existing_all.append(group_and_average_columns(load_pecd_onshore(30, y), COUNTRIES))
    onshore_new_all.append(group_and_average_columns(load_pecd_onshore(34, y), COUNTRIES))

onshore = 0.9 * pd.concat(onshore_new_all) + 0.1 * pd.concat(onshore_existing_all)
# Suite à des modifications du scénario nW, on a recalibré la courbe de charge pour la passer à 0.28
# Afin d'être applicable à d'autres scénarios d'émissions, on recalibre par la moyenne initiale.
onshore["FR"] = onshore["FR"] * 0.28 / 0.293598  # calage CLEVER sur la moyenne FR

onshore = save(onshore, "renew_profiles", "onshore")

onshore: 184104 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00


## Éolien offshore — offshore_float, offshore_ground

Deux fichiers PECD par année : moyenne européenne (`_20_`) et spécifique Italie (`_21_`).
La Suisse (CH) n'a pas de façade maritime : elle sera mise à 0 automatiquement par `save`.

- `offshore_ground` = moyenne européenne (avec IT remplacé par la valeur du fichier IT)
- `offshore_float` = mélange `0.7 * europe + 0.3 * IT`

In [59]:
def load_pecd_offshore(suffix, y):
    path = os.path.join(
        PECD_ROOT, "wind", "offshore",
        f"P_CMI6_MPI-_MEHR_WOF_NA---_Pecd_P2OF_S{y}01010000_E{y}12312300_CFR_TIM_01h_NA-_noc_org_{suffix}_SP{clim}_ReGrB_PhM04_PECD4.2_fv1.csv",
    )
    return pd.read_csv(path, index_col=0, parse_dates=True)


offshore_eu_all, offshore_it_all = [], []
for y in YEARS:
    eu = group_and_average_columns(load_pecd_offshore(20, y), COUNTRIES)
    it = group_and_average_columns(load_pecd_offshore(21, y), COUNTRIES)
    eu["IT"] = it["IT"]
    offshore_eu_all.append(eu)
    offshore_it_all.append(it)

offshore_eu = pd.concat(offshore_eu_all)
offshore_it = pd.concat(offshore_it_all)

offshore_float = 0.7 * offshore_eu + 0.3 * offshore_it
offshore_ground = offshore_eu

offshore_float = save(offshore_float, "renew_profiles", "offshore_float")
offshore_ground = save(offshore_ground, "renew_profiles", "offshore_ground")

offshore_float: 184104 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00
offshore_ground: 184104 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00


## Run-of-river — river

Génération hebdomadaire (HRO, `ror_generation`) sommée par pays, étalée à l'heure, convertie en
facteur de charge via une capacité installée de référence par pays. Correction manuelle IT (÷0.2, en
plus de la capacité de 6.5 GW) pour recaler sur la génération de référence.

In [60]:
river_all = []
for y in YEARS:
    river_y = group_and_sum_columns(read_pecd_hydro_series("ror_generation", "HRO", y), COUNTRIES)
    river_all.append(expand_weekly_to_hourly(river_y) / 1000)  # MW -> GW

river = pd.concat(river_all)
year_end = pd.Timestamp(f"{YEARS[-1]}-12-31 23:00:00")
river = extend_to_yearend(river, year_end)
river = river[river.index.year >= YEARS[0]]

max_potential = {"FR": 7.27, "DE": 3.55, "CH": 5.5, "IT": 7.8, "UK": 1.47, "BE": 0.1, "ES": 2, "NL": 0.04, "IE": 0.08, "PT": 0.8}  # GW
for c, cap in max_potential.items():
    river[c] = river[c] / cap
river["IT"] = river["IT"] / 1  # correction manuelle de calage

river = save(river, "renew_profiles", "river")

river: 184104 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00


## Lake inflows (réservoir) — lake_inflows

Inflows hebdomadaires du réservoir (HRI, `reservoir_inflow`), sommés par pays, étalés à l'heure, en GW.
BE, UK, NL, IE n'ont pas ce type de stockage dans le modèle : mis à 0.

In [61]:
lake_inflows_all = []
for y in YEARS:
    li_y = group_and_sum_columns(read_pecd_hydro_series("reservoir_inflow", "HRI", y), COUNTRIES)
    lake_inflows_all.append(expand_weekly_to_hourly(li_y) / 1000)  # MW -> GW

lake_inflows = pd.concat(lake_inflows_all)
lake_inflows = extend_to_yearend(lake_inflows, year_end)
lake_inflows = lake_inflows[lake_inflows.index.year >= YEARS[0]]
for c in ["BE", "UK", "NL", "IE"]:
    lake_inflows[c] = 0

lake_inflows = save(lake_inflows, "hydro_profiles", "lake_inflows")

lake_inflows: 184104 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00


## PHS inflows — phs_inflows

Même logique que `lake_inflows`, à partir des fichiers `phs_inflow` (HOL). BE, UK, NL, IE mis à 0.

In [62]:
phs_inflows_all = []
for y in YEARS:
    phs_y = group_and_sum_columns(read_pecd_hydro_series("phs_inflow", "HOL", y), COUNTRIES)
    phs_inflows_all.append(expand_weekly_to_hourly(phs_y) / 1000)  # MW -> GW

phs_inflows = pd.concat(phs_inflows_all)
phs_inflows = extend_to_yearend(phs_inflows, year_end)
phs_inflows = phs_inflows[phs_inflows.index.year >= YEARS[0]]
for c in ["BE", "UK", "NL", "IE"]:
    phs_inflows[c] = 0

phs_inflows = save(phs_inflows, "hydro_profiles", "phs_inflows")

phs_inflows: 184104 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00


## Contraintes de lacs — lake_minimal_outflow, lake_minimal_volume, lake_spill

Ces contraintes ne sont documentées que pour la France sur une année de référence (2006, pas
disponible dans PECD). Les autres pays sont mis à 0, puis le profil FR 2006 est recopié sur
2040-2060. `lake_minimal_outflow` est au pas journalier (contrainte du modèle), les deux autres au
pas horaire.

In [63]:
lake_constraints = {
    "lake_minimal_outflow": ("production profiles/lake inflows and constraints/lake_minimal_outflow.csv", "daily"),
    "lake_minimal_volume": ("production profiles/lake inflows and constraints/lake_minimal_volume.csv", "hourly"),
    "lake_spill": ("production profiles/lake inflows and constraints/lake_spill_2006.csv", "hourly"),
}

lake_constraint_results = {}
for name, (path, scale) in lake_constraints.items():
    fr_2006 = fr_only_other_zero(fr_only_to_countries(path, COUNTRIES, years=[2006], time_scale=scale))
    tiled = pd.concat([change_year(fr_2006, y) for y in YEARS])
    lake_constraint_results[name] = save(tiled, "hydro_profiles", name)

lake_minimal_outflow: 7665 lignes, 2040-01-01 00:00:00 -> 2060-12-31 00:00:00


lake_minimal_volume: 183960 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00
lake_spill: 183960 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00


## Autres renouvelables — marine, biomass_coge, geothermal_coge, ocgt_coge, waste

Profil France 2006 répliqué à l'identique sur tous les pays (pas de zéro : contrairement aux
contraintes de lacs, ces technologies existent partout dans le modèle), puis dupliqué sur 2040-2060.

In [64]:
other_renewables = {
    "marine": "production profiles/marine/marine_profile_1y.csv",
    "biomass_coge": "production profiles/biomass_coge/biomass_coge_profile_1y.csv",
    "geothermal_coge": "production profiles/geothermal_coge/geothermal_coge_profile_1y.csv",
    "ocgt_coge": "production profiles/ocgt_coge/ocgt_coge_profile_1y.csv",
    "waste": "production profiles/waste/waste_profile_1y.csv",
}

other_renewables_results = {}
for name, path in other_renewables.items():
    fr_2006 = fr_only_to_countries(path, COUNTRIES, years=[2006], time_scale="hourly")
    tiled = pd.concat([change_year(fr_2006, y) for y in YEARS])
    other_renewables_results[name] = save(tiled, "renew_profiles", name)

marine: 183960 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00
biomass_coge: 183960 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00
geothermal_coge: 183960 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00
ocgt_coge: 183960 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00
waste: 183960 lignes, 2040-01-01 00:00:00 -> 2060-12-31 23:00:00


## Vérification finale

Relit chaque CSV écrit et vérifie qu'il couvre bien 2040-01-01 à 2060-12-31 avec les 10 colonnes pays.

In [65]:
for subfolder in ["renew_profiles", "hydro_profiles"]:
    folder = os.path.join(OUTPUT_DIR, subfolder)
    for fname in sorted(os.listdir(folder)):
        df = pd.read_csv(os.path.join(folder, fname), index_col=0, parse_dates=True)
        missing = set(COUNTRIES) - set(df.columns)
        print(f"{fname:28s} {df.index.min()} -> {df.index.max()}  cols_manquantes={missing or 'aucune'}")

biomass_coge.csv             2040-01-01 00:00:00 -> 2060-12-31 23:00:00  cols_manquantes=aucune
geothermal_coge.csv          2040-01-01 00:00:00 -> 2060-12-31 23:00:00  cols_manquantes=aucune
marine.csv                   2040-01-01 00:00:00 -> 2060-12-31 23:00:00  cols_manquantes=aucune
ocgt_coge.csv                2040-01-01 00:00:00 -> 2060-12-31 23:00:00  cols_manquantes=aucune
offshore_float.csv           2040-01-01 00:00:00 -> 2060-12-31 23:00:00  cols_manquantes=aucune
offshore_ground.csv          2040-01-01 00:00:00 -> 2060-12-31 23:00:00  cols_manquantes=aucune
onshore.csv                  2040-01-01 00:00:00 -> 2060-12-31 23:00:00  cols_manquantes=aucune
pv_ground.csv                2040-01-01 00:00:00 -> 2060-12-31 23:00:00  cols_manquantes=aucune
pv_roof_com.csv              2040-01-01 00:00:00 -> 2060-12-31 23:00:00  cols_manquantes=aucune
pv_roof_indiv.csv            2040-01-01 00:00:00 -> 2060-12-31 23:00:00  cols_manquantes=aucune
river.csv                    2040-01-01 

## Check cohérence

In [94]:
CF_370 = pd.read_csv(f"created_time_varying_inputs/SSP370/renew_profiles/pv_ground.csv", index_col = 0, parse_dates = True)
CF_245 = pd.read_csv(f"created_time_varying_inputs/SSP245/renew_profiles/pv_ground.csv", index_col = 0, parse_dates = True)

In [95]:
CF_370.mean()

FR    0.133763
DE    0.113075
CH    0.131819
IT    0.156826
NL    0.109680
BE    0.112237
ES    0.170643
PT    0.170627
UK    0.102802
IE    0.100491
dtype: float64

In [96]:
CF_245.mean()

FR    0.135328
DE    0.116022
CH    0.134190
IT    0.158198
NL    0.111839
BE    0.114857
ES    0.170510
PT    0.170456
UK    0.103492
IE    0.099781
dtype: float64